<a href="https://colab.research.google.com/github/BilalKhaliqWillis/BILAL-Assignment2/blob/main/BILAL_Project_6_Human_Activity_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Human Activity Recognition (HAR) using LSTM

# Upload and Extract Dataset

In [1]:
from google.colab import files
uploaded = files.upload()

import zipfile
import os

for file_name in uploaded.keys():
    if file_name.endswith('.zip'):
        with zipfile.ZipFile(file_name, 'r') as zip_ref:
            zip_ref.extractall()

print("Dataset extracted successfully!")

Saving UCI HAR Dataset.zip to UCI HAR Dataset.zip
Dataset extracted successfully!


# Import Required Libraries

In [2]:
import numpy as np
import pandas as pd
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Load Dataset

In [3]:
def load_X(path):
    files = os.listdir(path)
    data = []
    for file in sorted(files):
        data.append(np.loadtxt(os.path.join(path, file)))
    return np.dstack(data)

def load_y(path):
    y = np.loadtxt(path)
    return y - 1

X_train = load_X('UCI HAR Dataset/train/Inertial Signals/')
X_test = load_X('UCI HAR Dataset/test/Inertial Signals/')

y_train = load_y('UCI HAR Dataset/train/y_train.txt')
y_test = load_y('UCI HAR Dataset/test/y_test.txt')

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

Training shape: (7352, 128, 9)
Test shape: (2947, 128, 9)


# Preprocess Data

In [4]:
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

print("y_train shape:", y_train.shape)

y_train shape: (7352, 6)


# Build LSTM Model

In [5]:
model = Sequential()

model.add(LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True))
model.add(Dropout(0.5))

model.add(LSTM(32))
model.add(Dropout(0.5))

model.add(Dense(64, activation='relu'))
model.add(Dense(6, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128, 64)        │        18,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,862 (132.27 KB)

 Trainable params: 33,862 (132.27 KB)

 Non-trainable params: 0 (0.00 B)

# Train Model

In [6]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/10
115/115 ━━━━━━━━━━━━━━━━━━━━ 24s 171ms/step - accuracy: 0.4861 - loss: 1.2318 - val_accuracy: 0.5914 - val_loss: 0.9648
Epoch 2/10
115/115 ━━━━━━━━━━━━━━━━━━━━ 22s 192ms/step - accuracy: 0.6712 - loss: 0.7800 - val_accuracy: 0.6820 - val_loss: 0.8405
Epoch 3/10
115/115 ━━━━━━━━━━━━━━━━━━━━ 19s 168ms/step - accuracy: 0.6766 - loss: 0.6847 - val_accuracy: 0.7153 - val_loss: 0.6791
Epoch 4/10
115/115 ━━━━━━━━━━━━━━━━━━━━ 21s 180ms/step - accuracy: 0.7613 - loss: 0.5349 - val_accuracy: 0.7479 - val_loss: 0.6023
Epoch 5/10
115/115 ━━━━━━━━━━━━━━━━━━━━ 40s 172ms/step - accuracy: 0.7524 - loss: 0.5577 - val_accuracy: 0.7496 - val_loss: 0.5402
Epoch 6/10
115/115 ━━━━━━━━━━━━━━━━━━━━ 21s 181ms/step - accuracy: 0.7916 - loss: 0.4352 - val_accuracy: 0.7618 - val_loss: 0.5158
Epoch 7/10
115/115 ━━━━━━━━━━━━━━━━━━━━ 19s 169ms/step - accuracy: 0.8112 - loss: 0.4192 - val_accuracy: 0.8035 - val_loss: 0.5013
Epoch 8/10
115/115 ━━━━━━━━━━━━━━━━━━━━ 21s 181ms/step - accuracy: 0.8672 - loss: 0

# Evaluate Model

In [7]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

print("Accuracy:", accuracy_score(y_true, y_pred_classes))
print("\nClassification Report:\n", classification_report(y_true, y_pred_classes))
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred_classes))

93/93 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step
Accuracy: 0.8924329826942654

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.94      0.90       496
           1       0.88      0.85      0.86       471
           2       0.99      1.00      0.99       420
           3       0.87      0.74      0.80       491
           4       0.79      0.89      0.84       532
           5       1.00      0.95      0.97       537

    accuracy                           0.89      2947
   macro avg       0.90      0.89      0.89      2947
weighted avg       0.90      0.89      0.89      2947


Confusion Matrix:
 [[468  27   1   0   0   0]
 [ 69 400   2   0   0   0]
 [  0   2 418   0   0   0]
 [  7   0   0 362 122   0]
 [  4   0   0  56 472   0]
 [  0  27   0   0   0 510]]


# Simulated Real-Time Prediction

In [8]:
def predict_activity(sample):
    sample = sample.reshape(1, sample.shape[0], sample.shape[1])
    prediction = model.predict(sample)
    return np.argmax(prediction)

sample = X_test[0]
predicted_class = predict_activity(sample)

print("Predicted Activity Class:", predicted_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Predicted Activity Class: 4
